# Boundary-Aware Inpainting Pipeline

**Before running anything:**
1. Runtime → Change runtime type → **T4 GPU** (free) or **A100** (Pro)
2. Run cells top to bottom in order
3. Run Cell 7 (smoke test) FIRST before Cell 8 (full pipeline)

In [ ]:
# ── CELL 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── CELL 2: Install dependencies ───────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'lpips', 'einops', 'scikit-image', 'clean-fid', 'timm'], check=True)
print('Dependencies installed.')

In [ ]:
# ── CELL 3: Extract code from Drive ────────────────────────────────────────
import zipfile, os, sys, shutil

# Path to your zip on Drive
ZIP_PATH = '/content/drive/MyDrive/boundary_inpainting_final_v4.zip'
CODE_DIR  = '/content/final_colab'

# Remove old version if exists
if os.path.exists(CODE_DIR):
    shutil.rmtree(CODE_DIR)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')

os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print('Code extracted. Files:', os.listdir(CODE_DIR))

In [ ]:
# ── CELL 4: Symlink checkpoints + results to Drive ─────────────────────────
import os

CODE_DIR   = '/content/final_colab'
DRIVE_ROOT = '/content/drive/MyDrive/boundary_inpainting'

def symlink_to_drive(drive_path, local_path):
    os.makedirs(drive_path, exist_ok=True)
    if os.path.islink(local_path): os.unlink(local_path)
    elif os.path.isdir(local_path): import shutil; shutil.rmtree(local_path)
    os.symlink(drive_path, local_path)
    print(f'  {local_path} → {drive_path}')

print('Symlinking to Drive:')
symlink_to_drive(f'{DRIVE_ROOT}/checkpoints', f'{CODE_DIR}/checkpoints')
symlink_to_drive(f'{DRIVE_ROOT}/results',     f'{CODE_DIR}/results')
print('Done. Checkpoints will survive disconnects.')

In [ ]:
# ── CELL 5: Point to your dataset on Drive ─────────────────────────────────
import os

CODE_DIR   = '/content/final_colab'
DRIVE_ROOT = '/content/drive/MyDrive/boundary_inpainting'

# Your Places365 images are already on Drive.
# Update PLACES_PATH to wherever your images are.
PLACES_PATH = f'{DRIVE_ROOT}/datasets/places365/val_256'

# Verify images exist
imgs = [f for f in os.listdir(PLACES_PATH) if f.endswith(('.jpg','.jpeg','.png'))]
print(f'Found {len(imgs)} images at: {PLACES_PATH}')
print(f'First 3: {imgs[:3]}')

In [ ]:
# ── CELL 6: GPU check ──────────────────────────────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {vram:.1f} GB')
    if vram >= 35:
        print('A100 detected — use batch_size=48')
    elif vram >= 15:
        print('T4 detected — use batch_size=16')
    else:
        print('Small GPU — use batch_size=8')

In [ ]:
# ── CELL 7: SMOKE TEST ─────────────────────────────────────────────────────
# Runs ~5-10 minutes on T4. Tests the full pipeline on tiny data.
# Run this BEFORE the full pipeline.
# If exit code = 0: proceed to Cell 8.
# If exit code = 1: paste the error output here and fix before proceeding.

import subprocess, sys, os
CODE_DIR    = '/content/final_colab'
PLACES_PATH = '/content/drive/MyDrive/boundary_inpainting/datasets/places365/val_256'
os.chdir(CODE_DIR)

result = subprocess.run(
    [
        sys.executable, 'main.py',
        '--profile',      'smoke',
        '--places_dir',   PLACES_PATH,
        '--skip_phases123',
        '--skip_multiseed',
        '--skip_convnext',
        '--num_workers',  '2',
    ],
    cwd=CODE_DIR,
    capture_output=True,
    text=True,
)

# Print last 100 lines so you can see progress + any errors
lines = (result.stdout + result.stderr).split('\n')
print('\n'.join(lines[-100:]))
print(f'\nSmoke test exit code: {result.returncode}')
if result.returncode == 0:
    print('SMOKE TEST PASSED — run Cell 8 for full pipeline')
else:
    print('SMOKE TEST FAILED — check errors above before proceeding')

In [ ]:
# ── CELL 8: FULL PIPELINE ──────────────────────────────────────────────────
# Only run after smoke test passes.
#
# Time estimates:
#   T4  GPU (free)  : ~40-50 hours total (use skip flags below)
#   A100 GPU (Pro+) : ~17-29 hours total
#
# With --skip_phases123 --skip_multiseed --skip_convnext:
#   T4  : ~25-30 hours
#   A100: ~17 hours
#
# If Colab disconnects: just re-run this cell.
# It auto-resumes from _resume.pt checkpoints — no training lost.

import subprocess, sys, os
CODE_DIR    = '/content/final_colab'
PLACES_PATH = '/content/drive/MyDrive/boundary_inpainting/datasets/places365/val_256'
os.chdir(CODE_DIR)

result = subprocess.run(
    [
        sys.executable, 'main.py',
        '--profile',      'production',
        '--places_dir',   PLACES_PATH,
        '--num_workers',  '4',
        '--skip_phases123',   # uses resnet34+attn_skip+gated_conv (proven best)
        '--skip_multiseed',   # only seed 42 (saves ~8 hours)
        '--skip_convnext',    # skip ConvNeXt runs (saves ~4 hours)
    ],
    cwd=CODE_DIR,
)
print(f'\nPipeline exit code: {result.returncode}')

In [ ]:
# ── CELL 9: Download results ────────────────────────────────────────────────
# Results are already on Drive. Download locally as backup too.
import shutil
from google.colab import files

shutil.make_archive('/content/results', 'zip', '/content/final_colab/results')
files.download('/content/results.zip')
print('Done. Also check Drive: boundary_inpainting/results/')